In [3]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

final_df = pd.read_parquet(processed_path / "final_dataset.parquet")

In [4]:
"SUBJECT_ID" in final_df.columns

False

In [5]:
cohort_df = pd.read_parquet("../data/processed/adult_icu_cohort_first24h.parquet")

cohort_df.columns

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'INTIME'], dtype='str')

In [6]:
patient_map = cohort_df[["ICUSTAY_ID", "SUBJECT_ID"]]

In [7]:
patient_map.head()

,ICUSTAY_ID,SUBJECT_ID
0,280836,268
1,206613,269
2,220345,270
3,249196,271
4,210407,272


In [8]:
final_df = final_df.merge(
    patient_map,
    on="ICUSTAY_ID",
    how="left"
)

In [9]:
final_df[["ICUSTAY_ID", "SUBJECT_ID"]].head()

,ICUSTAY_ID,SUBJECT_ID
0,280836,268
1,206613,269
2,220345,270
3,249196,271
4,210407,272


In [10]:
final_df["SUBJECT_ID"].isna().sum()

np.int64(0)

In [11]:
y = final_df["HOSPITAL_EXPIRE_FLAG"]

subject_id = final_df["SUBJECT_ID"]

In [12]:
X = final_df.drop(columns=[
    "HOSPITAL_EXPIRE_FLAG",
    "SUBJECT_ID",
    "ICUSTAY_ID"
])

In [16]:
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

In [ ]:
train_idx, test_idx = next(
    splitter.split(X, y, groups=subject_id)
)

In [ ]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

## Train-Test Split

The final dataset was prepared for machine learning by separating predictors, target, and patient identifiers.

- `HOSPITAL_EXPIRE_FLAG` was defined as the target variable.
- `SUBJECT_ID` was used only to keep multiple ICU stays from the same patient in the same split.
- `ICUSTAY_ID` and `SUBJECT_ID` were excluded from model predictors.
- The data was split into approximately 80% training and 20% test sets.
- `GroupShuffleSplit` was used to prevent ICU stays from the same patient from appearing in both training and test sets.